# Benchmark Results — H200

Per-component timing breakdown across Qwen3-0.6B, Qwen3-1.7B, and Qwen3-8B.

In [ ]:
import json
import glob
import os
import pandas as pd
from collections import defaultdict
from IPython.display import display, HTML, Markdown

RESULTS_DIR = os.path.join(os.getcwd(), "results", "h200")
if not os.path.isdir(RESULTS_DIR):
    RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "SFT", "benchmark", "results", "h200")
print(f"Results dir: {RESULTS_DIR}")
print(f"JSON files: {len(glob.glob(os.path.join(RESULTS_DIR, '*.json')))}")

In [ ]:
# =============================================================================
# Loading & helpers
# =============================================================================

METHODS = ["standard", "layerwise", "subset", "subset_one_pass"]
SCORINGS = ["compress", "ghost_greats", "ghost", "direct"]
METHOD_DISPLAY = {
    "standard": "Standard",
    "layerwise": "Layerwise",
    "subset": "Subset 2P",
    "subset_one_pass": "Subset 1P",
}

def load_all_results(results_dir):
    """Load all JSON result files, grouped by model tag.
    Returns: {(tag, model_name): {config_label: {method/scoring: result_dict}}}
    """
    models = defaultdict(dict)
    for path in sorted(glob.glob(os.path.join(results_dir, "*.json"))):
        basename = os.path.basename(path)
        parts = basename.replace(".json", "").rsplit("_", 3)
        if len(parts) < 4:
            continue
        tag = parts[0]
        try:
            with open(path) as f:
                data = json.load(f)
        except (json.JSONDecodeError, IOError):
            continue
        model_name = data.get("model", tag)
        for label, combos in data.get("results", {}).items():
            models[(tag, model_name)][label] = combos
    return dict(models)


def compute_total(r, method):
    if r is None:
        return None
    if method in ("standard", "layerwise"):
        return r.get("forward", 0) + r.get("backward", 0) + r.get("optimizer", 0)
    elif method == "subset":
        return (r.get("pass1_forward", 0) + r.get("pass1_backward", 0) +
                r.get("selection", 0) + r.get("pass2_forward", 0) +
                r.get("pass2_backward", 0) + r.get("optimizer", 0))
    elif method == "subset_one_pass":
        return (r.get("forward", 0) + r.get("backward", 0) +
                r.get("selection", 0) + r.get("wgrad", 0) +
                r.get("optimizer", 0))
    return None


def get_score_cost(r):
    if r is None:
        return None
    return r.get("score", 0) + r.get("compress", 0) + r.get("p1_score", 0) + r.get("p1_compress", 0)


def parse_config(label):
    parts = {}
    for tok in label.split():
        k, v = tok.split("=")
        parts[k] = int(v)
    return parts.get("n"), parts.get("T"), parts.get("m")


def sort_configs(configs):
    return sorted(configs, key=lambda l: parse_config(l))


all_models = load_all_results(RESULTS_DIR)
for (tag, model_name), configs in sorted(all_models.items()):
    n_ok = sum(sum(1 for v in c.values() if v is not None) for c in configs.values())
    print(f"{tag}: {len(configs)} configs, {n_ok} successful combos")

## 1. Subset 1P Overhead vs Standard

In [ ]:
def overhead_table(tag_filter=None):
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        if tag_filter and tag_filter not in tag:
            continue
        for label in sort_configs(configs):
            combos = configs[label]
            std_r = combos.get("standard/ghost")
            std_total = compute_total(std_r, "standard") if std_r else None
            if std_total is None:
                continue
            row = {"Model": tag, "Config": label, "Std (ms)": f"{std_total:.0f}"}
            for scoring in SCORINGS:
                r = combos.get(f"subset_one_pass/{scoring}")
                total = compute_total(r, "subset_one_pass")
                if total is not None:
                    row[scoring] = f"+{(total / std_total - 1) * 100:.1f}%"
                else:
                    row[scoring] = "OOM"
            rows.append(row)
    return pd.DataFrame(rows)

df = overhead_table()
display(df.style.set_caption("Subset 1P Overhead vs Standard"))

## 2. Score Cost (ms)

In [ ]:
def score_cost_table(method_prefix="subset_one_pass", tag_filter=None):
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        if tag_filter and tag_filter not in tag:
            continue
        for label in sort_configs(configs):
            combos = configs[label]
            row = {"Model": tag, "Config": label}
            for scoring in SCORINGS:
                r = combos.get(f"{method_prefix}/{scoring}")
                cost = get_score_cost(r)
                row[scoring] = f"{cost:.1f}" if cost is not None else "OOM"
            rows.append(row)
    return pd.DataFrame(rows)

df = score_cost_table()
display(df.style.set_caption("Score Cost (ms) — Subset 1P"))

## 3. One-Pass Speedup vs Two-Pass

In [ ]:
def onepass_speedup_table(tag_filter=None):
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        if tag_filter and tag_filter not in tag:
            continue
        for label in sort_configs(configs):
            combos = configs[label]
            row = {"Model": tag, "Config": label}
            for scoring in SCORINGS:
                t_2p = compute_total(combos.get(f"subset/{scoring}"), "subset")
                t_1p = compute_total(combos.get(f"subset_one_pass/{scoring}"), "subset_one_pass")
                if t_2p and t_1p and t_2p > 0:
                    row[scoring] = f"{(1 - t_1p / t_2p) * 100:.1f}%"
                else:
                    row[scoring] = "—"
            rows.append(row)
    return pd.DataFrame(rows)

df = onepass_speedup_table()
display(df.style.set_caption("One-Pass Speedup vs Two-Pass"))

## 4. Peak Memory (GB)

In [ ]:
def memory_table(tag_filter=None):
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        if tag_filter and tag_filter not in tag:
            continue
        for label in sort_configs(configs):
            combos = configs[label]
            std_r = combos.get("standard/ghost")
            std_mem = std_r.get("peak_memory_gb", 0) if std_r else None
            row = {"Model": tag, "Config": label,
                   "Standard": f"{std_mem:.1f}" if std_mem else "—"}
            for method, disp in [("layerwise", "Layerwise"), ("subset", "Subset 2P"), ("subset_one_pass", "Subset 1P")]:
                mems = [combos.get(f"{method}/{s}", {}).get("peak_memory_gb")
                        for s in SCORINGS if combos.get(f"{method}/{s}") is not None]
                mems = [m for m in mems if m is not None]
                row[disp] = f"{min(mems):.1f}–{max(mems):.1f}" if mems else "—"
            rows.append(row)
    return pd.DataFrame(rows)

df = memory_table()
display(df.style.set_caption("Peak Memory (GB)"))

## 5. Ghost vs Ghost_greats Crossover

In [ ]:
# Model dimensions for V* computation
MODEL_DIMS = {
    "qwen3-0.6b": (1024, 3072),
    "qwen3-1.7b": (2048, 6144),
    "qwen3-8b":   (4096, 12288),
}

def crossover_table():
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        O, I = MODEL_DIMS.get(tag, (0, 0))
        for label in sort_configs(configs):
            combos = configs[label]
            n, T, m = parse_config(label)
            r_gg = combos.get("subset_one_pass/ghost_greats")
            r_g = combos.get("subset_one_pass/ghost")
            cost_gg = get_score_cost(r_gg)
            cost_g = get_score_cost(r_g)
            if cost_gg is None and cost_g is None:
                continue
            vstar = O * I / (T * (O + I)) if O and I and T else None
            if cost_gg is not None and cost_g is not None:
                ratio = cost_gg / cost_g
                if abs(ratio - 1) < 0.1:
                    winner = "~tie"
                elif cost_gg < cost_g:
                    winner = "ghost_greats"
                else:
                    winner = "ghost"
            else:
                winner = "—"
            predicted = "ghost_greats" if (vstar and m < vstar) else "ghost" if vstar else "—"
            rows.append({
                "Model": tag, "Config": label,
                "ghost_gr (ms)": f"{cost_gg:.0f}" if cost_gg else "OOM",
                "ghost (ms)": f"{cost_g:.0f}" if cost_g else "OOM",
                "V*": f"{vstar:.1f}" if vstar else "—",
                "Predicted": predicted,
                "Actual": winner,
                "Match": "Y" if winner == predicted or winner == "~tie" else "N",
            })
    return pd.DataFrame(rows)

df = crossover_table()
def highlight_match(val):
    if val == "Y":
        return "background-color: #d4edda"
    elif val == "N":
        return "background-color: #f8d7da"
    return ""
display(df.style.applymap(highlight_match, subset=["Match"]).set_caption("Ghost vs Ghost_greats Crossover Validation"))

## 6. Detailed Breakdown (per model, per config)

In [ ]:
def breakdown_table(tag_filter=None, config_filter=None):
    """Full per-component breakdown. Filter by tag and/or config label."""
    rows = []
    for (tag, model_name), configs in sorted(all_models.items()):
        if tag_filter and tag_filter not in tag:
            continue
        for label in sort_configs(configs):
            if config_filter and config_filter not in label:
                continue
            combos = configs[label]
            std_r = combos.get("standard/ghost")
            std_total = compute_total(std_r, "standard") if std_r else None

            for method in METHODS:
                scorings = ["ghost"] if method == "standard" else SCORINGS
                for scoring in scorings:
                    key = f"{method}/{scoring}"
                    r = combos.get(key)
                    total = compute_total(r, method)
                    if r is None or total is None:
                        rows.append({"Model": tag, "Config": label,
                                     "Method": METHOD_DISPLAY.get(method, method),
                                     "Scoring": scoring if method != "standard" else "--",
                                     "Total": "OOM", "Overhead": ""})
                        continue

                    overhead = f"+{(total / std_total - 1) * 100:.1f}%" if std_total and method != "standard" else "--"
                    row = {
                        "Model": tag, "Config": label,
                        "Method": METHOD_DISPLAY.get(method, method),
                        "Scoring": scoring if method != "standard" else "--",
                        "forward": r.get("forward", r.get("pass1_forward", 0)),
                        "act_grad": r.get("act_grad", r.get("p1_act_grad", 0)),
                        "compress": r.get("compress", r.get("p1_compress", 0)) or None,
                        "score": r.get("score", r.get("p1_score", 0)) or None,
                        "select": r.get("select", 0) or None,
                        "w.grad": r.get("wgrad", 0),
                        "pass2_fwd": r.get("pass2_forward", 0) or None,
                        "pass2_bwd": r.get("pass2_backward", 0) or None,
                        "optim": r.get("optimizer", 0),
                        "Total": f"{total:.1f}",
                        "Overhead": overhead,
                        "Mem (GB)": f"{r.get('peak_memory_gb', 0):.1f}",
                    }
                    rows.append(row)
    df = pd.DataFrame(rows)
    # Format numeric columns
    for col in ["forward", "act_grad", "compress", "score", "select", "w.grad",
                "pass2_fwd", "pass2_bwd", "optim"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{x:.1f}" if pd.notna(x) and x else "")
    return df

# Show one model at a time to avoid huge tables
for (tag, _) in sorted(all_models.keys()):
    df = breakdown_table(tag_filter=tag)
    if not df.empty:
        display(Markdown(f"### {tag}"))
        display(df.style.set_caption(f"Detailed Breakdown — {tag}"))

## 7. Quick Comparison Across Models

Compare Subset 1P total times across all 3 models at matched (T, m) settings.

In [ ]:
def cross_model_comparison():
    """For each (T, m) that exists across multiple models, show totals side by side."""
    # Collect: {(T, m): {tag: {scoring: total}}}
    grid = defaultdict(lambda: defaultdict(dict))
    for (tag, _), configs in all_models.items():
        for label in configs:
            n, T, m = parse_config(label)
            combos = configs[label]
            std_r = combos.get("standard/ghost")
            std_total = compute_total(std_r, "standard") if std_r else None
            grid[(T, m)][tag]["standard"] = std_total
            for scoring in SCORINGS:
                r = combos.get(f"subset_one_pass/{scoring}")
                total = compute_total(r, "subset_one_pass")
                grid[(T, m)][tag][scoring] = total

    rows = []
    tags = sorted(set(tag for (tag, _) in all_models.keys()))
    for (T, m) in sorted(grid.keys()):
        for scoring in ["standard"] + SCORINGS:
            row = {"T": T, "m": m, "Method": f"1P/{scoring}" if scoring != "standard" else "standard"}
            for tag in tags:
                val = grid[(T, m)].get(tag, {}).get(scoring)
                row[tag] = f"{val:.0f}" if val else "—"
            rows.append(row)
    return pd.DataFrame(rows)

df = cross_model_comparison()
if not df.empty:
    display(df.style.set_caption("Cross-Model: Total Step Time (ms) at matched (T, m)"))